# 7.2 트리 하나에서 숲으로: 랜덤 포레스트 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter07_2_random_forest.ipynb)

책 본문: [7.2 트리 하나에서 숲으로: 랜덤 포레스트](https://smhanlab.com/book-ml/kor/ml1/chapter07/2.html)

이 노트북은 책 7.2절의 모든 수치를 코드로 재현합니다:

1. 깊이가 3인 결정 트리가 어떤 질문을 던지는지(유방암 데이터셋)
2. 부트스트랩 샘플 5개로 학습한 개별 트리 5개의 test 정확도가 얼마나 흩어지는지
3. 랜덤 포레스트(100개)가 그보다 정확도도 높고, 시드에 대한 안정성도 좋은지
4. 2D 장난감 데이터에서 "트리 하나"와 "랜덤 포레스트"의 결정 경계를 눈으로 비교
5. 테스트 세트 없이도 일반화 성능을 가질 수 있는 OOB(out-of-bag) 점수
6. 트리 개수(`n_estimators`)를 늘릴수록 정확도·안정성이 어떻게 수렴하는지
7. 배깅(특징 무작위화 없음)과 랜덤 포레스트(특징 무작위화 있음)의 차이


In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier

IMG = "/home/smhan/book-ml/kor/src/images"   # (Colab에서는 /tmp로 바꾸면 됨)


## 1. 깊이 3짜리 트리: 어떤 질문을 던지는가

본문과 같은 설정 — 유방암 진단 데이터셋(양성/악성), train/test 7:3, 깊이 3.
알고리즘이 정보이득이 가장 큰 질문(`worst concave points`)을 루트에
자동으로 놓는지 확인합니다.


In [2]:
data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.3, random_state=0)

tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train, y_train)
print(export_text(tree, feature_names=list(data.feature_names), max_depth=2))
print("train acc:", round(tree.score(X_train, y_train), 3))
print("test acc: ", round(tree.score(X_test, y_test), 3))


|--- worst concave points <= 0.14
|   |--- worst area <= 952.90
|   |   |--- area error <= 35.26
|   |   |   |--- class: 1
|   |   |--- area error >  35.26
|   |   |   |--- class: 1
|   |--- worst area >  952.90
|   |   |--- mean symmetry <= 0.15
|   |   |   |--- class: 1
|   |   |--- mean symmetry >  0.15
|   |   |   |--- class: 0
|--- worst concave points >  0.14
|   |--- area error <= 13.93
|   |   |--- class: 1
|   |--- area error >  13.93
|   |   |--- worst perimeter <= 79.13
|   |   |   |--- class: 1
|   |   |--- worst perimeter >  79.13
|   |   |   |--- class: 0

train acc: 0.967
test acc:  0.947


## 2. 배깅의 재료: 부트스트랩 트리가 얼마나 흩어지는가

"서로 다른 실수를 하는 전문가들의 평균이 더 안정적이다"라는 주장을
숫자로 확인하기 위한 1단계: 학습 데이터에서 **복원추출**(부트스트랩)로
원본과 같은 크기의 샘플 5개를 뽑고, 각각으로 **깊이 제한 없는**(=
완전히 자라나는, 과적합하는) 트리를 독립적으로 학습시킨 뒤 test
정확도를 잽니다.

7.1절에서 봤듯 완전한 트리는 학습 데이터는 외우지만 새 데이터에
취약합니다. 트리마다 본 "데이터"가 조금씩 다를 뿐인데, 정확도가
얼마나 폭으로 흩어지는지가 "개별 트리의 불안정성"을 그대로
보여줍니다.


In [3]:
rng = np.random.RandomState(0)
boots = []
for t in range(5):
    idx = rng.choice(len(X_train), size=len(X_train), replace=True)
    btree = DecisionTreeClassifier(max_depth=None, random_state=t).fit(X_train[idx], y_train[idx])
    acc = btree.score(X_test, y_test)
    boots.append(acc)
    print(f"tree {t}: test acc = {acc:.3f}")

print("mean:", round(np.mean(boots), 3),
      " std(pct):", round(100 * np.std(boots), 2))


tree 0: test acc = 0.871
tree 1: test acc = 0.918
tree 2: test acc = 0.895
tree 3: test acc = 0.889
tree 4: test acc = 0.883
mean: 0.891  std(pct): 1.55


## 3. 랜덤 포레스트: 정확도 상승 + 시드 안정성

(2)에서 보듯 개별(완전한) 트리 하나로는 정확도가 제법 흔들립니다.
같은 데이터를 100개 트리로 배깅하면 어떻게 되는지, 그리고
랜덤 포레스트를 서로 다른 무작위 시드로 다시 만들어봐도
성능이 크게 흔들리지 않는지 확인합니다.


In [4]:
rf = RandomForestClassifier(n_estimators=100, random_state=0).fit(X_train, y_train)
print("random forest (100 trees):", round(rf.score(X_test, y_test), 3))
print("(개별 5개 트리의 최댓값:", round(max(boots), 3), ")")

accs = []
for s in range(5):
    m = RandomForestClassifier(n_estimators=100, random_state=s).fit(X_train, y_train)
    accs.append(m.score(X_test, y_test))
    print(f"seed {s}: {accs[-1]:.3f}")
print("range:", round(min(accs), 3), "-", round(max(accs), 3),
      " std(pct):", round(100 * np.std(accs), 2))


random forest (100 trees): 0.959
(개별 5개 트리의 최댓값: 0.918 )
seed 0: 0.959


seed 1: 0.965


seed 2: 0.977
seed 3: 0.959


seed 4: 0.971
range: 0.959 - 0.977  std(pct): 0.68


## 4. 2D 장난감 데이터: 결정 경계를 눈으로 비교

유방암 데이터는 30개 특징이라 2D로 그릴 수 없습니다.
그래서 2개 특징짜리 작은 데이터(두 덩어리)를 만들어,
"트리 하나(깊이 3)"와 "랜덤 포레스트(100개)"의 결정 경계를
그려봅니다.

- 왼쪽(단일 트리): 각자 직선 조각(축에 평행한 경계)이 이어진 **계단형** 경계.
- 오른쪽(랜덤 포레스트): 계단이 다수결로 평균되어 **훨씬 부드러운** 경계.

이 부드러움(= 분산 감소)이 랜덤 포레스트가 단일 트리보다
일반화가 좋은 이유의 시각화입니다.


In [5]:
from sklearn.datasets import make_blobs

X2, y2 = make_blobs(n_samples=300, cluster_std=1.2,
                    centers=[(-2, -1), (2, 2)], random_state=0)
tr = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X2, y2)
rf2 = RandomForestClassifier(n_estimators=100, random_state=0).fit(X2, y2)
print("2D train acc: tree", round(tr.score(X2, y2), 3), " rf", round(rf2.score(X2, y2), 3))

xs = np.linspace(X2[:, 0].min() - 0.5, X2[:, 0].max() + 0.5, 200)
ys = np.linspace(X2[:, 1].min() - 0.5, X2[:, 1].max() + 0.5, 200)
XX, YY = np.meshgrid(xs, ys)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
for ax, clf, name in [(axes[0], tr, "Decision tree (depth 3)"),
                      (axes[1], rf2, "Random forest (100 trees)")]:
    Z = clf.predict(np.c_[XX.ravel(), YY.ravel()]).reshape(XX.shape)
    ax.contourf(XX, YY, Z, levels=[-0.5, 0.5, 1.5], alpha=0.15, cmap="coolwarm")
    ax.contour(XX, YY, Z, levels=[0.5], colors="k", linewidths=1.0)
    ax.scatter(X2[y2 == 0, 0], X2[y2 == 0, 1], s=18, c="b", alpha=0.6, label="class 0")
    ax.scatter(X2[y2 == 1, 0], X2[y2 == 1, 1], s=18, c="r", alpha=0.6, label="class 1")
    ax.set_title(name)
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")
    ax.legend(fontsize=8)
fig.suptitle("Decision boundary: one tree vs random forest")
fig.tight_layout()
fig.savefig(IMG + "/ch07_2_boundary_tree_vs_rf.svg")
print("SVG 저장: kor/src/images/ch07_2_boundary_tree_vs_rf.svg")


2D train acc: tree 0.973  rf 1.0
SVG 저장: kor/src/images/ch07_2_boundary_tree_vs_rf.svg


## 5. 테스트 세트 없이도: OOB(out-of-bag) 점수

각 트리의 부트스트랩 샘플은 약 \(37\%\)의 원본 데이터(복원추출에서
떨어져 나오는 샘플)를 자동으로 "보지 못"합니다. 이 트리들을 합쳐서
그 트리가 보지 못한 샘플을 예측하면, **학습 중** 일반화 성능을
평가할 수 있습니다 — 검증 세트를 따로 떼어놓을 필요가 없습니다.
`oob_score=True` 옵션으로 켜면 됩니다.


In [6]:
rf_oob = RandomForestClassifier(n_estimators=100, oob_score=True,
                                random_state=0).fit(X_train, y_train)
print("OOB score:", round(rf_oob.oob_score_, 3))
print("test score:", round(rf_oob.score(X_test, y_test), 3))
print("(OOB가 test와 비슷하게 붙는 것을 확인 — 별도 검증 없이도 '이 모델이 과적합인가'를 가질 수 있음)")


OOB score: 0.955
test score: 0.959
(OOB가 test와 비슷하게 붙는 것을 확인 — 별도 검증 없이도 '이 모델이 과적합인가'를 가질 수 있음)


## 6. 트리 개수(`n_estimators`)를 늘리면

배깅의 효과는 트리를 늘릴수록 (수확체감으로) 계속 좋아지고,
중요한 것은 **흔들림(분산)도 줄어든다**는 점입니다.
`n_estimators`를 1→200까지 늘리면서, 각각 10개 무작위 시드로
평균한 test 정확도와 표준편차를 그립니다.
(GBDT와 달리 여기서 정확도가 **떨어지는** 지점이 없는 이유를
본문 "자주 하는 실수" 섹션에서 설명합니다.)


In [7]:
ns = [1, 2, 5, 10, 20, 50, 100, 200]
means, stds = [], []
for n in ns:
    accs_n = [RandomForestClassifier(n_estimators=n, random_state=s,
                                     n_jobs=-1).fit(X_train, y_train).score(X_test, y_test)
              for s in range(10)]
    means.append(np.mean(accs_n))
    stds.append(np.std(accs_n))
    print(f"n={n:4d}: mean={np.mean(accs_n):.3f} std={np.std(accs_n):.3f}")

plt.figure(figsize=(6.5, 4))
plt.errorbar(ns, means, yerr=stds, fmt="o-", capsize=3, label="test acc (10 seeds)")
plt.xscale("log")
plt.xlabel("n_estimators (log scale)")
plt.ylabel("test accuracy")
plt.title("Random forest: accuracy vs number of trees")
plt.ylim(0.85, 1.0)
plt.grid(alpha=0.3)
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(IMG + "/ch07_2_n_estimators_curve.svg")
print("SVG 저장: kor/src/images/ch07_2_n_estimators_curve.svg")


n=   1: mean=0.931 std=0.015


n=   2: mean=0.913 std=0.018


n=   5: mean=0.938 std=0.011


n=  10: mean=0.952 std=0.012


n=  20: mean=0.958 std=0.012


n=  50: mean=0.964 std=0.008


n= 100: mean=0.963 std=0.009


n= 200: mean=0.963 std=0.005


SVG 저장: kor/src/images/ch07_2_n_estimators_curve.svg


## 7. 특징 중요도: 숲 전체의 시선

각 트리가 분기마다 줄인 지니불순도를 특징별로 누적한 것
(7.1절의 `feature_importances_`)을 100개 트리에 걸쳐 평균하면,
"숲 전체"가 어떤 특징을 가장 많이 봤는지 알 수 있습니다.


In [8]:
imp = np.argsort(rf.feature_importances_)[::-1]
plt.figure(figsize=(7, 4))
plt.barh(range(5), rf.feature_importances_[imp[:5]][::-1],
         color="steelblue")
plt.yticks(range(5), [data.feature_names[i] for i in imp[:5]][::-1], fontsize=9)
plt.xlabel("importance (sum of Gini decrease, averaged over trees)")
plt.title("Random forest: top-5 feature importances (breast cancer)")
plt.tight_layout()
plt.savefig(IMG + "/ch07_2_feature_importance.svg")
print("SVG 저장: kor/src/images/ch07_2_feature_importance.svg")

for i in imp[:5]:
    print(f"  {data.feature_names[i]}: {rf.feature_importances_[i]:.3f}")


SVG 저장: kor/src/images/ch07_2_feature_importance.svg
  worst perimeter: 0.151
  worst concave points: 0.135
  mean concave points: 0.131
  worst radius: 0.091
  mean concavity: 0.087


## 8. 특징 무작위화의 역할: "강한 특징"이 루트를 독점하는가?

본문 FAQ(1번)의 질문 — "특징 무작위화가 꼭 필요한가" — 을 직접
확인합니다. `BaggingClassifier`는 트리를 **배깅만** 해줍니다
(분기 시점의 특징 무작위화 없음). 두 앙상블의 모든 트리를 만들고,
각 트리가 **루트에서** 어떤 특징을 사용하는지 셉니다.

`worst concave points`가 유난히 강한 특징이라, 배깅만 쓴
앙상블에서는 "부트스트랩 샘플이 달라져도" 루트에서 거의 항상
그 특징을 고르게 됩니다 → 트리들이 서로 닮아갑니다.
반면 랜덤 포레스트는 각 분기에서 전체 30개 특징 중 무작위로
고른 일부(기본값 `max_features=sqrt(30)≈5`)만 보므로, 그 강한
특징조차 "이번엔 후보에 안 들어간" 경우가 생겨 루트에서
덜 자주 쓰입니다 → 트리들이 서로 다른 질문을 탐색합니다.


In [9]:
from sklearn.ensemble import BaggingClassifier

dom = list(data.feature_names).index("worst concave points")
print("dominant feature:", data.feature_names[dom])

bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=6, random_state=0),
    n_estimators=50, random_state=0).fit(X_train, y_train)
rf8 = RandomForestClassifier(n_estimators=50, max_depth=6, random_state=0).fit(X_train, y_train)

def root_frac(clf):
    # 루트 노드를 dominant feature로 나누는 트리의 비율
    return np.mean([t.tree_.feature[0] == dom for t in clf.estimators_])

print("\nfrac of trees using the dominant feature at the ROOT:")
print(f"  bagging-only (no feature rand): {root_frac(bag):.2f}")
print(f"  random forest (feature rand):   {root_frac(rf8):.2f}")
print(f"\nRF max_features default = sqrt(30) = {np.sqrt(30):.2f}")
print("acc: bagging-only", round(bag.score(X_test, y_test), 3),
      " random forest", round(rf8.score(X_test, y_test), 3))
print("(랜덤 포레스트가 루트에서 강한 특징을 덜 독점함 -> 트리 다양성 증가)")


dominant feature: worst concave points



frac of trees using the dominant feature at the ROOT:
  bagging-only (no feature rand): 0.40
  random forest (feature rand):   0.20

RF max_features default = sqrt(30) = 5.48
acc: bagging-only 0.959  random forest 0.965
(랜덤 포레스트가 루트에서 강한 특징을 덜 독점함 -> 트리 다양성 증가)
